# SnoozeSound Pillow: Power Analysis Study Design

**AIPI 510 — Week 6 - Yongrong Lu**

## Client Claim

“Users who sleep with our embedded white noise pillow fall asleep 15 minutes faster on average than those without it.”

**Outcome:** sleep onset latency in minutes. This notebook proposes a study and calculates its sample size before data collection.

The primary question is whether SnoozeSound really reduces average sleep onset latency. Testing for any reduction is different from establishing the specific 15-minute claim. I also estimate the mean reduction and its confidence interval to assess the magnitude.

## 1. Study Design

Use a randomized controlled experiment with two independent groups and equal (1:1) allocation:

- **Treatment:** adults use the SnoozeSound pillow with its embedded white noise enabled.
- **Control:** adults use an otherwise comparable pillow with no white noise.

Each participant follows their assigned condition for seven nights. Define nightly sleep onset latency as minutes from lights out while attempting sleep to the first epoch of Stage N2 sleep, measured using the same sleep-recording and scoring protocol in both groups. Average the seven nightly values for each participant; each participant contributes one primary outcome. Prespecify how incomplete recordings will be handled before recruitment.

Keep bedtime instructions, recording equipment, and environmental conditions as consistent as practicable. Record adherence and use blinded sleep scoring where possible. Participants will hear the sound, so full participant blinding is unlikely; expectations remain a limitation.

**Rationale:** random allocation helps balance confounders, the matched control limits differences unrelated to sound, and a participant-level average avoids treating repeated nights as independent people. The N2 definition makes the outcome explicit, although it is narrower than the everyday meaning of “fall asleep.”

## 2. Null and Alternative Hypotheses

Let $\mu_S$ denote the population mean participant-level sleep onset latency under SnoozeSound and $\mu_C$ the corresponding control mean.

$$H_0: \mu_S \geq \mu_C$$

$$H_1: \mu_S < \mu_C$$

Equivalently, define the benefit $\Delta = \mu_C - \mu_S$: $H_0: \Delta \leq 0$ versus $H_1: \Delta > 0$.

**Rationale:** the hypothesis predicts faster sleep onset, so the one-sided direction is specified before examining data. An effect in the opposite direction will not count as evidence for the claim. Failure to reject the null does not prove that there is no effect.

## 3. Statistical Test and Assumptions

Use a one-sided independent-samples t-test, with a pooled-variance Student t-test as the planning model. It compares means for a continuous outcome in two groups of different participants.

1. **Independent observations:** each adult is assigned to one group and contributes one average. Recruitment should avoid clustering, such as multiple participants sharing the same sleep environment. Randomization alone does not eliminate dependence.
2. **Continuous outcome:** time in minutes is continuous and measured consistently across groups.
3. **Approximately normal participant-level outcomes:** inspect histograms and Q–Q plots for severe skew and influential outliers. Moderate samples offer some robustness, but substantial skew or extreme values could require a revised analysis and power calculation.
4. **Comparable population variances:** the pooled t-test and this power calculation assume a common within-group variance. Inspect group spreads; if materially unequal variances are expected, prespecify Welch's independent-samples t-test and reassess power under plausible group variances. Do not treat a nonsignificant variance test as proof of equality.

Report the control-minus-treatment mean difference and a confidence interval alongside the one-sided p-value. The present hypothesis tests a benefit above zero. Establishing a benefit of at least 15 minutes would require a different null boundary and its own power calculation.

## 4. Literature and Effect-Size Rationale

[Messineo et al. (2017)](https://doi.org/10.3389/fneur.2017.00718) studied 18 healthy adults in a randomized crossover experiment with two overnight recordings. Participants experienced environmental noise and room-delivered broadband sound in random order, with bedtime advanced to model transient insomnia. Median latency to Stage N2 was approximately **19 minutes in control and 13 minutes with sound** ($p = 0.011$); the authors reported a **38% median reduction**. That reported percentage is not the percentage obtained by simply dividing the difference between the two rounded group medians by 19.

This supports the direction of the proposed hypothesis, but the sample was small, the intervention used room speakers, and the design used repeated measurements on the same people. These results do not establish a 15-minute benefit from a pillow. Medians and a reported percentage alone do not supply the independent-group means and standard deviations needed to calculate the planned Cohen's d directly.

## 5. Choice of Effect Size

Use **Cohen's $d = 0.50$**, a conventional medium standardized mean difference, as an explicit planning assumption:

$$d = \frac{\mu_C - \mu_S}{\sigma_{\mathrm{pooled}}}.$$

A positive value represents a benefit because higher control latency means the SnoozeSound group falls asleep sooner. For observed data, the pooled standard deviation is

$$s_{\mathrm{pooled}} = \sqrt{\frac{(n_C-1)s_C^2+(n_S-1)s_S^2}{n_C+n_S-2}}.$$

I choose a medium effect rather than assuming a large effect transfers from earlier research. This is a cautious alternative to an optimistic large-effect assumption. If the true effect is smaller, the planned study will have less power.

The 15-minute claim does not itself imply $d=0.50$. A 15-minute difference would correspond to $d=0.50$ only if the relevant common standard deviation were **30 minutes** ($15/30=0.50$). I do not have evidence establishing that standard deviation. A pilot study could improve the planning assumptions.

## 6. A Priori Power Analysis

| Parameter | Choice | Rationale |
|---|---|---|
| Test | One-sided independent-samples t-test | Directional difference between two independent means |
| Cohen's d | 0.50 | Medium planning effect; not estimated directly from the cited study |
| Significance level, alpha | 0.05 | Conventional 5% Type I error threshold at the null boundary |
| Target power | 0.80 | 80% probability of detecting the assumed effect under the planning model |
| Allocation ratio | 1:1 | Balanced groups are efficient under a common variance and similar costs |

Use `statsmodels.stats.power.TTestIndPower`. Set group 1 to **control** and group 2 to **SnoozeSound** conceptually, so positive d and `alternative="larger"` mean control latency is greater. This is equivalent to a lower-tailed test when subtracting control from SnoozeSound.

**To run:** select a Python 3 kernel with `statsmodels` installed and run all cells in order. No dataset or network access is needed for these calculations.

In [1]:
import math
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()
effect_size = 0.50  # (mean_control - mean_snoozesound) / pooled_SD
alpha = 0.05
target_power = 0.80

sample_size = analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=target_power,
    ratio=1.0,
    alternative="larger",
)
n_per_group = math.ceil(sample_size)
total_n = 2 * n_per_group
achieved_power = analysis.power(
    effect_size=effect_size,
    nobs1=n_per_group,
    alpha=alpha,
    ratio=1.0,
    alternative="larger",
)

print(f"Required sample size per group: {sample_size:.2f}")
print(f"Rounded sample size per group: {n_per_group}")
print(f"Total required sample size: {total_n}")
print(f"Power at the rounded sample size: {achieved_power:.3f}")

Required sample size per group: 50.15
Rounded sample size per group: 51
Total required sample size: 102
Power at the rounded sample size: 0.806


### Interpretation

The calculation requires approximately 50.15 participants per group. Round upward to 51 per group, giving 102 participants with analyzable outcomes. Under the stated assumptions, this provides at least 80% power to detect d = 0.50 at a one-sided alpha of 0.05.



## 7. Optional Allowance for 10% Attrition

The 102-participant requirement refers to analyzable outcomes. If approximately 10% of participants are expected to lack such outcomes, divide the requirement by the anticipated retention rate and round upward **within each group**:

$$n_{\mathrm{recruit,group}} = \lceil 51 / (1-0.10) \rceil = 57.$$

Recruit **57 per group, or 114 total**. This allowance targets the expected retention level and it cannot guarantee 51 complete outcomes in each group. Prespecify missing-data handling, and do not automatically exclude participants simply because they did not adhere to treatment.

In [2]:
attrition_rate = 0.10
recruit_per_group = math.ceil(n_per_group / (1 - attrition_rate))
recruit_total = 2 * recruit_per_group
print(f"Recruitment target per group: {recruit_per_group}")
print(f"Recruitment target with 10% attrition: {recruit_total}")

Recruitment target per group: 57
Recruitment target with 10% attrition: 114


## 8. Final Study Recommendation

| Component | Decision |
|---|---|
| Design | Randomized, parallel, two independent groups |
| Treatment / control | White-noise pillow / comparable pillow without sound |
| Outcome | Participant-level mean latency to N2 sleep, in minutes |
| H0 / H1 | Mean SnoozeSound latency ≥ control / < control |
| Primary test | One-sided independent-samples t-test |
| Planning assumptions | Common variance, d = 0.50, alpha = 0.05, power = 0.80 |
| Required analyzable sample | **51 per group; 102 total** |
| Optional recruitment target | **57 per group; 114 total** with 10% attrition |

The design evaluates whether the intervention reduces mean sleep onset latency. The cited experiment motivates the direction, while d = 0.50 remains a planning assumption because its setting and design differ from this study. Interpret any future result using both its statistical significance and the estimated reduction in minutes; a positive result alone does not validate the advertised 15-minute magnitude.

## References

1. Messineo, L., Taranto-Montemurro, L., Sands, S. A., Oliveira Marques, M. D., Azabarzin, A., & Wellman, D. A. (2017). Broadband sound administration improves sleep onset latency in healthy subjects in a model of transient insomnia. *Frontiers in Neurology, 8*, 718. [https://doi.org/10.3389/fneur.2017.00718](https://doi.org/10.3389/fneur.2017.00718)
2. Cohen, J. (1988). *Statistical power analysis for the behavioral sciences* (2nd ed.). Lawrence Erlbaum Associates. (Conventional standardized effect-size benchmarks.)
3. statsmodels developers. *TTestIndPower.solve_power*. [Official API documentation](https://www.statsmodels.org/stable/generated/statsmodels.stats.power.TTestIndPower.solve_power.html). (Independent two-sample t-test power calculation and one-sided alternative.)